# Text2Cypher Retriever

Vector retrievers are great for finding relevant data based on semantic similarity, but sometimes you need to answer specific questions about your graph structure—questions that require precise Cypher queries.

**Use cases for Text2Cypher:**
- What asset manager owns a specific organization?
- How many stock types are there?
- What organizations are exposed to a certain risk factor?

The `Text2CypherRetriever` converts natural language queries into Cypher queries using an LLM, executes them against the graph, and returns structured results.

**Prerequisites:** You need an existing graph with entities (from Lab 5 or earlier labs).

---

**Learning Objectives:**
- Create a `Text2CypherRetriever` using Databricks Model Serving
- Generate and execute Cypher queries from natural language
- Build a `GraphRAG` pipeline with Text2Cypher retrieval

## Setup

In [ ]:
# Install dependencies for Databricks Model Serving
%pip install neo4j-graphrag databricks-langchain python-dotenv pydantic-settings nest-asyncio -q

In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.schema import get_schema

from data_utils import Neo4jConnection, get_llm, get_embedder

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

llm = get_llm()
embedder = get_embedder()

print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.endpoint}")

## Understanding the Graph Schema

The `Text2CypherRetriever` needs to understand your graph structure to generate valid Cypher queries. The `get_schema` function extracts node labels, relationship types, and properties from your Neo4j database.

In [ ]:
schema = get_schema(driver)
print(schema)

## Create the Text2CypherRetriever

The retriever uses an LLM to translate natural language questions into Cypher queries based on your schema.

**How it works:**
1. Takes your question and the graph schema
2. Uses the LLM to generate a Cypher query
3. Executes the query and returns results

We provide a custom prompt to ensure modern Cypher syntax (Neo4j 5+).

In [ ]:
# Custom prompt to ensure generated Cypher queries follow modern best practices
TEXT2CYPHER_PROMPT = """Task: Generate a Cypher statement to query a graph database.

Instructions:
- Use only the provided relationship types and properties in the schema.
- Do not use any other relationship types or properties that are not provided.
- Only filter by name when a specific entity name is mentioned in the question.
  When filtering by name, use case-insensitive matching:
  `WHERE toLower(node.name) CONTAINS toLower('ActualEntityName')`
- Do NOT add name filters if no specific entity name is mentioned in the question.
- Always add LIMIT 20 to the end of your query to restrict results.

Modern Cypher Requirements:
- Use `elementId(node)` instead of `id(node)` (id() is removed in Neo4j 5+).
- Use `count{{pattern}}` instead of `size((pattern))` for counting patterns.
- Use `EXISTS {{MATCH pattern}}` instead of `exists((pattern))` for existence checks.
- When using ORDER BY, filter NULL values first: `WHERE property IS NOT NULL ORDER BY property`.
- Use explicit grouping with WITH clauses for aggregations.
- Limit collected results when appropriate: `collect(item)[0..20]`.

Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

The question is:
{query_text}"""

text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=schema,
    custom_prompt=TEXT2CYPHER_PROMPT
)

print("Text2CypherRetriever initialized with Databricks LLM")

## Execute Natural Language Queries

The retriever automatically:
1. Generates a Cypher query using the LLM
2. Executes it against Neo4j
3. Returns both the generated Cypher and the results

Try asking questions about entities in your graph.

In [ ]:
query = "What companies are in the database?"
cypher_result = text2cypher_retriever.get_search_results(query)

print(f"Original Query: {query}")
print(f"Generated Cypher: {cypher_result.metadata['cypher']}")
print(f"Number of results: {len(cypher_result.records)}\n")

print("Results:")
for record in cypher_result.records[:5]:  # Show first 5
    print(f"  {record}")

In [ ]:
# Try a more specific query
query = "What products does Apple offer?"
cypher_result = text2cypher_retriever.get_search_results(query)

print(f"Original Query: {query}")
print(f"Generated Cypher: {cypher_result.metadata['cypher']}")
print(f"Number of results: {len(cypher_result.records)}\n")

print("Results:")
for record in cypher_result.records:
    print(f"  {record}")

## GraphRAG with Text2Cypher

Combine the Text2Cypher retriever with `GraphRAG` to generate natural language answers based on the query results.

In [ ]:
# Note: Text2CypherRetriever doesn't use top_k - result limiting is handled
# by the LIMIT clause in the generated Cypher query (set in the prompt).

rag = GraphRAG(llm=llm, retriever=text2cypher_retriever)

query = "What services does Apple provide?"
response = rag.search(query, return_context=True)

print(f"Query: {query}\n")
print("Answer:")
print(response.answer)

In [ ]:
# View the generated Cypher and context used
print("Generated Cypher:")
print(response.retriever_result.metadata["cypher"])
print(f"\nNumber of results: {len(response.retriever_result.items)}")
print("\nContext used:")
for item in response.retriever_result.items[:3]:
    print(f"  {item.content}")

## Experiment

Try different queries against your graph. The Text2Cypher retriever excels at:
- Counting entities ("How many products are there?")
- Finding specific relationships ("What companies are connected to Apple?")
- Aggregations ("Which company has the most products?")
- Filtering by properties ("Find all risk factors")

In [ ]:
# Try your own queries
your_query = "How many different entity types are in the graph?"

response = rag.search(your_query, return_context=True)
print(f"Query: {your_query}\n")
print(f"Generated Cypher: {response.retriever_result.metadata['cypher']}\n")
print(f"Answer: {response.answer}")

## Summary

You've learned how to use `Text2CypherRetriever` with Databricks Model Serving to:

1. **Extract graph schema** - Automatically get node labels, relationships, and properties
2. **Generate Cypher from natural language** - LLM translates questions to queries
3. **Execute and retrieve results** - Run queries and get structured data
4. **Build GraphRAG pipelines** - Combine retrieval with LLM generation

**When to use Text2Cypher vs Vector Retrieval:**
- **Text2Cypher**: Specific questions about entities, relationships, counts, aggregations
- **Vector Retrieval**: Semantic similarity search, finding related content
- **Best practice**: Combine both in a hybrid approach!

---

**Next:** [Entity Extraction](02_entity_extraction.ipynb) - Build knowledge graphs from unstructured text.

In [ ]:
neo4j.close()